# Protein-Compound Affinity: Kaggle Training

This notebook runs the leakage-aware baseline, optionally extracts frozen ProLLaMA features, trains the fusion head, evaluates it, and exports ONNX. Use a GPU accelerator for LLM extraction.

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Change this to the directory where the GitHub repository is attached.
PROJECT = Path('/kaggle/input/protein-ligand-affinity-repo')
if not (PROJECT / 'pyproject.toml').exists():
    candidates = list(Path('/kaggle/input').glob('**/pyproject.toml'))
    if not candidates:
        raise FileNotFoundError('Attach this repository as a Kaggle Dataset, then rerun.')
    PROJECT = candidates[0].parent

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT}[onnx,llm,analysis]'], check=True)
os.chdir(PROJECT)
print('Project:', PROJECT)

## Locate and profile the competition CSV

The repository expects `protein_sequence`, `compound_smiles`, and `label`. The primary split is cold-protein because proteins repeat heavily.

In [ ]:
from affinity.data import load_dataset, profile_dataset

csv_candidates = list(Path('/kaggle/input').glob('**/train.csv'))
DATA = next(path for path in csv_candidates if 'protein' in str(path).lower() or 'affinity' in str(path).lower())
frame = load_dataset(DATA)
profile_dataset(frame)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_frame = frame.assign(
    protein_length=frame.protein_sequence.str.len(),
    smiles_length=frame.compound_smiles.str.len(),
).sample(min(50_000, len(frame)), random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(plot_frame.label, bins=40, kde=True, ax=axes[0])
sns.histplot(plot_frame.protein_length, bins=50, ax=axes[1])
sns.histplot(plot_frame.smiles_length, bins=50, ax=axes[2])
axes[0].set_title('Target'); axes[1].set_title('Protein length'); axes[2].set_title('SMILES length')
plt.tight_layout()

## Baseline smoke run

Run this before expensive extraction. Copy the config to `/kaggle/working` and set its data path to the discovered CSV.

In [ ]:
import tomllib

baseline_config = Path('/kaggle/working/baseline.toml')
text = (PROJECT / 'configs/baseline.toml').read_text()
text = text.replace('data/sample_train.csv', str(DATA).replace('\\', '/'))
text = text.replace('artifacts/baseline', '/kaggle/working/artifacts/baseline')
baseline_config.write_text(text)
!affinity-train --config /kaggle/working/baseline.toml
!affinity-export-onnx --artifacts /kaggle/working/artifacts/baseline

## Optional ProLLaMA embedding extraction

The command deduplicates protein sequences, loads ProLLaMA in 4-bit mode, mean-pools its final hidden states, and saves one vector per unique protein. A Kaggle secret named `HF_TOKEN` may be required.

In [ ]:
RUN_PROLLAMA = False
if RUN_PROLLAMA:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    Path('/kaggle/working/features').mkdir(exist_ok=True)
    subprocess.run([
        'python', '-m', 'affinity.llm_embeddings',
        '--data', str(DATA),
        '--column', 'protein_sequence',
        '--model-id', 'GreatCaptainNemo/ProLLaMA',
        '--output', '/kaggle/working/features/prollama_embeddings.npz',
        '--batch-size', '1', '--max-length', '1536'
    ], check=True)

## Mol-LLaMA feature extraction

Mol-LLaMA is not a standalone generic text checkpoint: its official path uses MoleculeSTM, Uni-Mol, the blending module, and Q-Former with molecular graph and 3D conformer inputs. The repository adapter below pools the official Q-Former query tokens and caches one vector per unique compound.

Do not replace this with plain SMILES prompting and call it equivalent to Mol-LLaMA; that bypasses the model's defining 2D/3D architecture.

In [ ]:
RUN_MOL_LLAMA = False
if RUN_MOL_LLAMA:
    official_repo = Path('/kaggle/working/Mol-LLaMA')
    if not official_repo.exists():
        subprocess.run(['git', 'clone', 'https://github.com/DongkiKim95/Mol-LLaMA', str(official_repo)], check=True)
    # Install the official repository's documented PyTorch/PyG/OpenBabel environment first.
    subprocess.run([
        'affinity-mol-embeddings', '--data', str(DATA),
        '--official-repo', str(official_repo),
        '--output', '/kaggle/working/features/mol_llama_embeddings.npz',
        '--batch-size', '1', '--precision', 'bfloat16'
    ], check=True)

## Export encoders, extract exact ONNX caches, and train

The final experiment should build its caches with the same ONNX encoders used by the Space. Set `PROLLAMA_ONNX` to the downloaded repository produced by the converter.

In [ ]:
RUN_MOL_ONNX_EXPORT = False
if RUN_MOL_ONNX_EXPORT:
    subprocess.run([
        'affinity-export-mol-onnx',
        '--official-repo', '/kaggle/working/Mol-LLaMA',
        '--output', '/kaggle/working/artifacts/mol_llama_onnx',
        '--quantize'
    ], check=True)

PROLLAMA_ONNX = '/kaggle/input/prollama-onnx'
RUN_ONNX_CACHE_EXTRACTION = False
if RUN_ONNX_CACHE_EXTRACTION:
    subprocess.run([
        'affinity-extract-onnx', '--data', str(DATA),
        '--prollama-onnx', PROLLAMA_ONNX,
        '--mol-llama-onnx', '/kaggle/working/artifacts/mol_llama_onnx',
        '--protein-output', '/kaggle/working/features/prollama_embeddings.npz',
        '--molecule-output', '/kaggle/working/features/mol_llama_embeddings.npz'
    ], check=True)

RUN_LLM_FUSION = False
if RUN_LLM_FUSION:
    config_text = (PROJECT / 'configs/kaggle_llm.toml').read_text()
    config_text = config_text.replace('/kaggle/input/protein-compound-affinity/train.csv', str(DATA))
    Path('/kaggle/working/kaggle_llm.toml').write_text(config_text)
    subprocess.run(['affinity-train', '--config', '/kaggle/working/kaggle_llm.toml'], check=True)
    subprocess.run([
        'affinity-export-onnx', '--artifacts', '/kaggle/working/artifacts/llm_fusion'
    ], check=True)

print('Artifacts:', list(Path('/kaggle/working/artifacts').glob('**/*')))